# 02 — Merge & Build Feature Matrix
## SustAInable | Neighborhood Heat Illness Risk Prediction

---

**Purpose:** Merges the cleaned datasets explored in Phase 2 EDA notebooks into a single analysis-ready feature matrix at the ZCTA level. This is the foundation the XGBoost model will train on.

| Input | What It Contains |
|---|---|
| CDC PLACES | Chronic disease prevalence by ZCTA |
| CDC Heat & Health Index (HHI) | Structural vulnerability scores by ZCTA |
| ACS 5-Year Estimates | Poverty, housing, disability rates by ZCTA *(if available)* |

**Output:** `data/processed/sustainable_features_merged.csv`

**Author:** Nico Meyering, MPA | Equitech Futures Civic Tech Institute, CTI 2026

---

## Step 0: Imports and Setup

Run this cell first every time you open this notebook.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Try importing sklearn — needed for composite score in Step 6
try:
    from sklearn.preprocessing import MinMaxScaler
    SKLEARN_AVAILABLE = True
except ImportError:
    SKLEARN_AVAILABLE = False
    print("⚠️  sklearn not installed. Run: pip install scikit-learn")
    print("   Step 6 (composite score) will be skipped until sklearn is available.")

# Nicer displays
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.4f}'.format)

print('✅ Imports complete.')
print(f'pandas: {pd.__version__} | numpy: {np.__version__}')

## Step 1: File Paths

This cell sets up directory paths and scans your `data/raw/` and `data/processed/` folders so you can see exactly what files are available.

In [ ]:
# ── Directory paths ────────────────────────────────────────────────────────────
DATA_RAW       = Path("../data/raw")
DATA_PROCESSED = Path("../data/processed")
OUTPUTS        = Path("../outputs")

# Create these directories if they don't exist yet
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
OUTPUTS.mkdir(parents=True, exist_ok=True)

# ── What do we have? ───────────────────────────────────────────────────────────
print("📁  data/raw/")
raw_files = sorted(DATA_RAW.glob("*")) if DATA_RAW.exists() else []
if raw_files:
    for f in raw_files:
        if f.is_file():
            size_mb = f.stat().st_size / 1024 / 1024
            print(f"    {f.name}  ({size_mb:.1f} MB)")
else:
    print("    ⚠️  No files found. Download datasets per data/README.md")

print()
print("📁  data/processed/")
proc_files = sorted(DATA_PROCESSED.glob("*")) if DATA_PROCESSED.exists() else []
if proc_files:
    for f in proc_files:
        if f.is_file():
            size_mb = f.stat().st_size / 1024 / 1024
            print(f"    {f.name}  ({size_mb:.1f} MB)")
else:
    print("    (empty — outputs from this notebook will be saved here)")

## ⚙️ Manual File Path Override

If the auto-detection below doesn't find your files, set the paths manually here. **You only need to edit this cell if auto-detection fails.**

```python
# Example:
PLACES_FILE_OVERRIDE = Path("../data/raw/your_places_filename.csv")
HHI_FILE_OVERRIDE    = Path("../data/raw/your_hhi_filename.csv")
ACS_FILE_OVERRIDE    = Path("../data/raw/your_acs_filename.csv")
```

In [ ]:
# Set to None to use auto-detection (recommended).
# Set to a Path() if auto-detection fails.
PLACES_FILE_OVERRIDE = None
HHI_FILE_OVERRIDE    = None
ACS_FILE_OVERRIDE    = None

## Step 2: Load CDC PLACES

CDC PLACES contains chronic disease prevalence estimates for every ZCTA in the US. The raw file is in **long format** — one row per ZCTA per health measure. We will pivot it to wide format (one row per ZCTA) in Step 3.

> **Where to get it:** https://www.cdc.gov/places — download the ZCTA-level file.

In [ ]:
# ── Auto-detect CDC PLACES file ───────────────────────────────────────────────
PLACES_CANDIDATES = [
    "cdc_places_zcta.csv",
    "cdc_places.csv",
    "places_zcta_2023.csv",
    "places_zcta_2024.csv",
    "PLACES__Local_Data_for_Better_Health__ZCTA_Data.csv",
    "places_cleaned.csv",
    "places_processed.csv",
]

def find_file(candidates, override, search_dirs):
    if override is not None:
        if override.exists():
            print(f"✅ Using manual override: {override}")
            return override
        else:
            print(f"⚠️  Override path not found: {override}")
            return None
    for d in search_dirs:
        for name in candidates:
            p = d / name
            if p.exists():
                return p
    return None

places_path = find_file(
    PLACES_CANDIDATES,
    PLACES_FILE_OVERRIDE,
    [DATA_RAW, DATA_PROCESSED]
)

if places_path:
    print(f"✅ Found: {places_path.name}")
    places_raw = pd.read_csv(places_path, dtype=str)
    # Convert numeric columns from string
    for col in places_raw.columns:
        try:
            places_raw[col] = pd.to_numeric(places_raw[col])
        except (ValueError, TypeError):
            pass
    places_raw = places_raw.convert_dtypes()
    print(f"   Shape: {places_raw.shape[0]:,} rows × {places_raw.shape[1]} columns")
    print(f"   Columns: {list(places_raw.columns)}")
    display(places_raw.head(3))
else:
    places_raw = None
    print("⚠️  CDC PLACES file not found.")
    print("   Download from: https://www.cdc.gov/places")
    print("   Save to: data/raw/cdc_places_zcta.csv")
    print("   Then re-run this cell.")

## Step 3: Pivot CDC PLACES to Wide Format

Think of this like rotating a spreadsheet. Right now each health measure is its own row. We need each health measure to be its own **column**, with one row per ZCTA.

The measures we're extracting are the ones most predictive of heat illness vulnerability based on the SustAInable feature schema.

In [ ]:
# ── Target health measures ────────────────────────────────────────────────────
# These MeasureId values align with SustAInable's feature schema.
TARGET_MEASURES = [
    'DIABETES',    # Diagnosed diabetes — high heat illness risk factor
    'OBESITY',     # Obesity — heat intolerance amplifier
    'CHD',         # Coronary heart disease
    'COPD',        # COPD — respiratory vulnerability
    'BPHIGH',      # High blood pressure
    'CASTHMA',     # Asthma
    'KIDNEY',      # Chronic kidney disease
    'CSMOKING',    # Current smoking
    'DEPRESSION',  # Depression — social isolation risk
    'STROKE',      # Prior stroke
    'CANCER',      # Any cancer except skin
    'MHLTH',       # Mental health not good (past 30 days)
    'PHLTH',       # Physical health not good (past 30 days)
    'ACCESS2',     # Lack of health insurance
]

places_wide = None

if places_raw is not None:
    # ── Case 1: Long format with MeasureId column ──────────────────────────────
    if 'MeasureId' in places_raw.columns:
        available_measures = places_raw['MeasureId'].unique().tolist()
        found   = [m for m in TARGET_MEASURES if m in available_measures]
        skipped = [m for m in TARGET_MEASURES if m not in available_measures]

        print(f"✅ Measures found ({len(found)}):  {found}")
        if skipped:
            print(f"ℹ️  Measures not in file ({len(skipped)}): {skipped}")

        # Filter to crude prevalence + target measures
        filter_mask = places_raw['MeasureId'].isin(found)
        if 'DataValueTypeID' in places_raw.columns:
            filter_mask = filter_mask & (places_raw['DataValueTypeID'] == 'CrudePrevalence')

        places_filtered = places_raw[filter_mask].copy()

        # Identify the ZCTA column
        zcta_col_candidates = ['LocationID', 'ZCTA5', 'ZCTA', 'ZIP', 'LocationName']
        zcta_col = next((c for c in zcta_col_candidates if c in places_filtered.columns), None)
        value_col = 'Data_Value' if 'Data_Value' in places_filtered.columns else 'DataValue'

        if zcta_col and value_col in places_filtered.columns:
            places_wide = places_filtered.pivot_table(
                index=zcta_col,
                columns='MeasureId',
                values=value_col,
                aggfunc='mean'
            ).reset_index()
            places_wide.columns.name = None
            places_wide = places_wide.rename(columns={zcta_col: 'ZCTA'})
            # Prefix all measure columns
            measure_cols = [c for c in places_wide.columns if c != 'ZCTA']
            places_wide = places_wide.rename(
                columns={c: f'places_{c.lower()}' for c in measure_cols}
            )
            print(f"\n✅ Pivoted to wide format.")
            print(f"   Shape: {places_wide.shape[0]:,} ZCTAs × {places_wide.shape[1]} columns")
            display(places_wide.head(3))
        else:
            print(f"⚠️  Could not identify ZCTA column or value column.")
            print(f"   Available columns: {list(places_filtered.columns)}")

    # ── Case 2: Already in wide format ────────────────────────────────────────
    else:
        print("ℹ️  MeasureId column not found — file may already be in wide format.")
        print(f"   Columns: {list(places_raw.columns)}")
        zcta_col_candidates = ['LocationID', 'ZCTA5', 'ZCTA', 'ZIP']
        zcta_col = next((c for c in zcta_col_candidates if c in places_raw.columns), None)
        if zcta_col:
            places_wide = places_raw.rename(columns={zcta_col: 'ZCTA'}).copy()
            print(f"✅ Using as-is. ZCTA column: '{zcta_col}'")
        else:
            print(f"⚠️  Could not identify ZCTA column. Please check column names above.")

## Step 4: Load CDC Heat & Health Index (HHI)

The HHI gives each ZCTA a structural vulnerability score based on factors like elderly population, outdoor workers, and lack of air conditioning. It is a baseline vulnerability measure — SustAInable must outperform it.

> **Where to get it:** https://ephtracking.cdc.gov/Applications/heatTracker/

In [ ]:
HHI_CANDIDATES = [
    "cdc_hhi_zcta.csv",
    "hhi_zcta.csv",
    "heat_health_index.csv",
    "hhi_cleaned.csv",
    "cdc_heat_health_index.csv",
    "heat_vulnerability_index.csv",
    "hvi_zcta.csv",
    "heat_vulnerability_cleaned.csv",
]

hhi_path = find_file(
    HHI_CANDIDATES,
    HHI_FILE_OVERRIDE,
    [DATA_RAW, DATA_PROCESSED]
)

hhi_wide = None

if hhi_path:
    print(f"✅ Found: {hhi_path.name}")
    hhi_raw = pd.read_csv(hhi_path, dtype=str)
    for col in hhi_raw.columns:
        try:
            hhi_raw[col] = pd.to_numeric(hhi_raw[col])
        except (ValueError, TypeError):
            pass

    print(f"   Shape: {hhi_raw.shape[0]:,} rows × {hhi_raw.shape[1]} columns")
    print(f"   Columns: {list(hhi_raw.columns)}")

    # Identify ZCTA column
    zcta_col_candidates = ['ZCTA5', 'ZCTA', 'ZIP', 'ZCTA5CE10', 'zip_code', 'ZipCode']
    zcta_col = next((c for c in zcta_col_candidates if c in hhi_raw.columns), None)

    if zcta_col:
        hhi_wide = hhi_raw.rename(columns={zcta_col: 'ZCTA'}).copy()
        # Prefix all non-ZCTA columns
        feature_cols = [c for c in hhi_wide.columns if c != 'ZCTA']
        hhi_wide = hhi_wide.rename(
            columns={c: f'hhi_{c.lower()}' for c in feature_cols}
        )
        print(f"\n✅ HHI loaded. ZCTA column was: '{zcta_col}'")
        display(hhi_wide.head(3))
    else:
        print(f"\n⚠️  Could not identify ZCTA column in HHI file.")
        print(f"   Available columns: {list(hhi_raw.columns)}")
        print(f"   Set the ZCTA column name manually in the cell below.")
else:
    print("⚠️  HHI file not found.")
    print("   Download from: https://ephtracking.cdc.gov/Applications/heatTracker/")
    print("   Save to: data/raw/cdc_hhi_zcta.csv")
    print("   The merge will proceed without HHI features for now.")

## Step 5: Load ACS 5-Year Estimates *(Optional)*

ACS data adds poverty rate, housing age, renter rate, and disability prevalence. **If you don't have this file yet, this cell will skip gracefully** — the merge will proceed with PLACES + HHI. Come back to add ACS data in a later iteration.

> **Where to get it:** https://data.census.gov — search for ACS 5-Year ZCTA-level data.

In [ ]:
ACS_CANDIDATES = [
    "acs_zcta.csv",
    "acs_5year_zcta.csv",
    "acs_cleaned.csv",
    "census_acs.csv",
    "acs_features.csv",
]

acs_path = find_file(
    ACS_CANDIDATES,
    ACS_FILE_OVERRIDE,
    [DATA_RAW, DATA_PROCESSED]
)

acs_wide = None

if acs_path:
    print(f"✅ Found: {acs_path.name}")
    acs_raw = pd.read_csv(acs_path, dtype=str)
    for col in acs_raw.columns:
        try:
            acs_raw[col] = pd.to_numeric(acs_raw[col])
        except (ValueError, TypeError):
            pass

    print(f"   Shape: {acs_raw.shape[0]:,} rows × {acs_raw.shape[1]} columns")
    print(f"   Columns: {list(acs_raw.columns)}")

    zcta_col_candidates = ['ZCTA5', 'ZCTA', 'ZIP', 'GEOID', 'zip_code']
    zcta_col = next((c for c in zcta_col_candidates if c in acs_raw.columns), None)

    if zcta_col:
        acs_wide = acs_raw.rename(columns={zcta_col: 'ZCTA'}).copy()
        feature_cols = [c for c in acs_wide.columns if c != 'ZCTA']
        acs_wide = acs_wide.rename(
            columns={c: f'acs_{c.lower()}' for c in feature_cols}
        )
        print(f"\n✅ ACS loaded. ZCTA column was: '{zcta_col}'")
        display(acs_wide.head(3))
    else:
        print(f"\n⚠️  Could not identify ZCTA column. Columns: {list(acs_raw.columns)}")
else:
    print("ℹ️  ACS file not found — skipping.")
    print("   This is fine. Merge will proceed with CDC PLACES + HHI.")
    print("   Add ACS data later to enrich the feature matrix.")

## Step 6: Standardize Join Keys

All datasets must use the same ZCTA format before we can merge them. ZCTA codes are 5-digit strings — but they're often stored as integers, which silently drops leading zeros for ZCTAs starting with 0 (common in the Northeast).

> **Example:** ZCTA `01002` in Massachusetts gets stored as integer `1002` — and then the merge fails silently, because `"01002" ≠ "1002"`. This cell catches that before it causes problems.

In [ ]:
def standardize_zcta(df, label):
    """Force ZCTA column to 5-digit zero-padded string."""
    if df is None:
        return None
    df = df.copy()
    before = df['ZCTA'].head(3).tolist()
    df['ZCTA'] = (
        df['ZCTA']
        .astype(str)
        .str.strip()
        .str.zfill(5)          # pad to 5 digits with leading zeros
        .str.replace(r'\.0$', '', regex=True)   # strip ".0" from float conversions
    )
    after = df['ZCTA'].head(3).tolist()
    short = (df['ZCTA'].str.len() != 5).sum()
    print(f"✅ {label}: ZCTA standardized")
    print(f"   Before: {before}  →  After: {after}")
    if short > 0:
        print(f"   ⚠️  {short} ZCTAs still not exactly 5 characters — review these.")
    else:
        print(f"   All {len(df):,} ZCTAs are exactly 5 characters.")
    return df

places_wide = standardize_zcta(places_wide, "CDC PLACES")
hhi_wide    = standardize_zcta(hhi_wide,    "CDC HHI")
acs_wide    = standardize_zcta(acs_wide,    "ACS")

## Step 7: Merge Datasets

We use CDC PLACES as the base table because it has the broadest ZCTA coverage nationally. HHI and ACS data are left-joined onto it — meaning every PLACES ZCTA is kept, even if HHI or ACS doesn't have a matching row.

> **Why left join?** A ZCTA with missing HHI data isn't a reason to exclude it from the model. We handle the gaps in Step 8. Dropping ZCTAs at the merge stage would introduce geographic bias against areas with incomplete federal data — exactly the communities SustAInable is designed to catch.

In [ ]:
if places_wide is None:
    print("⚠️  CDC PLACES not loaded. Cannot build feature matrix.")
    print("   Complete Steps 2–3 first.")
    merged = None
else:
    merged = places_wide.copy()
    print(f"Base (CDC PLACES): {merged.shape[0]:,} ZCTAs × {merged.shape[1]} columns")

    if hhi_wide is not None:
        before = merged.shape[1]
        merged = merged.merge(hhi_wide, on='ZCTA', how='left', suffixes=('', '_hhi_dup'))
        # Drop accidental duplicate columns
        dup_cols = [c for c in merged.columns if c.endswith('_hhi_dup')]
        merged = merged.drop(columns=dup_cols)
        added = merged.shape[1] - before
        matched = hhi_wide['ZCTA'].isin(places_wide['ZCTA']).sum()
        print(f"After HHI join:    {merged.shape[0]:,} ZCTAs × {merged.shape[1]} columns  (+{added} features)")
        print(f"   HHI match rate: {matched:,} of {len(hhi_wide):,} HHI ZCTAs matched")
    else:
        print("ℹ️  HHI not loaded — skipped.")

    if acs_wide is not None:
        before = merged.shape[1]
        merged = merged.merge(acs_wide, on='ZCTA', how='left', suffixes=('', '_acs_dup'))
        dup_cols = [c for c in merged.columns if c.endswith('_acs_dup')]
        merged = merged.drop(columns=dup_cols)
        added = merged.shape[1] - before
        print(f"After ACS join:    {merged.shape[0]:,} ZCTAs × {merged.shape[1]} columns  (+{added} features)")
    else:
        print("ℹ️  ACS not loaded — skipped.")

    print(f"\n✅ Merge complete.")
    display(merged.head(3))

## Step 8: Assess Merge Quality

Before touching any data, we need to know exactly how much is missing and why. This step produces a quality report. Think of it as a pre-flight checklist.

In [ ]:
if merged is not None:
    print("=" * 60)
    print("MERGE QUALITY REPORT")
    print("=" * 60)
    print(f"Total ZCTAs:      {len(merged):,}")
    print(f"Total columns:    {merged.shape[1]}")

    null_counts = merged.isnull().sum()
    null_pct    = (null_counts / len(merged) * 100).round(2)
    null_report = pd.DataFrame({
        'null_count': null_counts,
        'null_%': null_pct
    }).sort_values('null_%', ascending=False)

    has_nulls = null_report[null_report['null_count'] > 0]
    print(f"\nColumns with missing data: {len(has_nulls)} of {merged.shape[1]}")
    if len(has_nulls) > 0:
        display(has_nulls.head(25))
    else:
        print("✅ No missing values — perfect merge!")

    # Visual: missing data heatmap
    feature_cols = [c for c in merged.columns if c != 'ZCTA']
    sample_cols  = feature_cols[:30]  # show up to 30 columns

    fig, ax = plt.subplots(figsize=(14, 6))
    null_matrix = merged[sample_cols].isnull()
    sns.heatmap(null_matrix.T, ax=ax, cbar=False,
                yticklabels=True, xticklabels=False,
                cmap=['#2ecc71', '#e74c3c'])
    ax.set_title('Missing Data Map — Red = Missing, Green = Present\n(Sample of first 30 feature columns)', 
                 fontsize=12)
    ax.set_xlabel(f'ZCTAs (n={len(merged):,})', fontsize=10)
    plt.tight_layout()
    plt.savefig(OUTPUTS / '02a_missing_data_map.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("✅ Chart saved to outputs/02a_missing_data_map.png")

## Step 9: Handle Missing Values

**Strategy:**

| Feature Group | Missing Value Treatment | Rationale |
|---|---|---|
| CDC PLACES (chronic disease) | Impute with national median | Suppressed small-area values, not true absence |
| HHI features | Impute median + add `_missing` flag | Model should know data was absent |
| ACS features | Impute median + add `_missing` flag | Same rationale |

The missingness indicator columns (e.g., `hhi_score_missing = 1`) let the model learn whether data absence itself is predictive — which it often is for underserved ZCTAs.

In [ ]:
if merged is not None:
    places_feature_cols = [c for c in merged.columns if c.startswith('places_')]
    hhi_feature_cols    = [c for c in merged.columns if c.startswith('hhi_')]
    acs_feature_cols    = [c for c in merged.columns if c.startswith('acs_')]

    imputed_count = 0

    # CDC PLACES — median imputation only (no missing flag)
    for col in places_feature_cols:
        if merged[col].isnull().sum() > 0:
            median_val = merged[col].median()
            merged[col] = merged[col].fillna(median_val)
            imputed_count += 1

    # HHI — missing flag + median imputation
    for col in hhi_feature_cols:
        if merged[col].isnull().sum() > 0:
            merged[f'{col}_missing'] = merged[col].isnull().astype(int)
            merged[col] = merged[col].fillna(merged[col].median())
            imputed_count += 1

    # ACS — missing flag + median imputation
    for col in acs_feature_cols:
        if merged[col].isnull().sum() > 0:
            merged[f'{col}_missing'] = merged[col].isnull().astype(int)
            merged[col] = merged[col].fillna(merged[col].median())
            imputed_count += 1

    remaining_nulls = merged.isnull().sum().sum()
    print(f"✅ Imputation complete.")
    print(f"   Columns imputed:    {imputed_count}")
    print(f"   Missingness flags added for HHI + ACS features")
    print(f"   Remaining nulls:    {remaining_nulls}")
    print(f"   Total columns now:  {merged.shape[1]}")

## Step 10: Feature Engineering

Two engineered features are added here:

1. **`composite_vulnerability_score`** — A normalized 0–1 score summarizing overall neighborhood vulnerability across chronic disease and structural factors. This is *not* a model input — it's a human-readable summary for stakeholder reporting.

2. **`multi_condition_burden`** — Count of how many chronic conditions are above their national median in a given ZCTA. A simple but useful signal.

> Both of these will appear in SHAP explanations once the model is trained, helping community organizations understand *why* a ZIP code was flagged.

In [ ]:
if merged is not None:
    # ── Composite vulnerability score ─────────────────────────────────────────
    # Candidate features for composite score
    vuln_keywords = ['diabetes', 'obesity', 'chd', 'copd', 'bphigh',
                     'casthma', 'kidney', 'poverty', 'elderly', 'disability',
                     'no_ac', 'renter', 'stroke', 'depression']

    vuln_feature_cols = [
        c for c in merged.columns
        if any(kw in c.lower() for kw in vuln_keywords)
        and not c.endswith('_missing')
        and pd.api.types.is_numeric_dtype(merged[c])
    ]

    if vuln_feature_cols and SKLEARN_AVAILABLE:
        scaler = MinMaxScaler()
        scaled = scaler.fit_transform(merged[vuln_feature_cols])
        merged['composite_vulnerability_score'] = scaled.mean(axis=1).round(4)
        print(f"✅ composite_vulnerability_score built from {len(vuln_feature_cols)} features:")
        for f in vuln_feature_cols:
            print(f"   · {f}")
        print()
        print(merged['composite_vulnerability_score'].describe().round(4))
    elif not SKLEARN_AVAILABLE:
        print("⚠️  sklearn not available — composite score skipped.")
        print("   Install with: pip install scikit-learn")
    else:
        print("⚠️  No matching vulnerability columns found for composite score.")
        print(f"   Available columns: {[c for c in merged.columns if c != 'ZCTA']}")

    # ── Multi-condition burden count ──────────────────────────────────────────
    condition_cols = [c for c in merged.columns
                      if c.startswith('places_')
                      and not c.endswith('_missing')]

    if condition_cols:
        medians = merged[condition_cols].median()
        above_median = (merged[condition_cols] > medians).astype(int)
        merged['multi_condition_burden'] = above_median.sum(axis=1)
        print(f"\n✅ multi_condition_burden built from {len(condition_cols)} condition columns.")
        print(merged['multi_condition_burden'].describe().round(2))

## Step 11: Feature Matrix Visualizations

Three views of the final dataset before saving.

In [ ]:
if merged is not None:
    feature_cols     = [c for c in merged.columns if c != 'ZCTA']
    places_cols      = [c for c in merged.columns if c.startswith('places_') and not c.endswith('_missing')]
    hhi_cols         = [c for c in merged.columns if c.startswith('hhi_') and not c.endswith('_missing')]
    acs_cols         = [c for c in merged.columns if c.startswith('acs_') and not c.endswith('_missing')]
    engineered_cols  = [c for c in merged.columns if c in ['composite_vulnerability_score', 'multi_condition_burden']]

    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    fig.suptitle('SustAInable — Feature Matrix Overview', fontsize=14, fontweight='bold')

    # 1. Feature count by source
    source_counts = {
        'CDC PLACES': len(places_cols),
        'HHI': len(hhi_cols),
        'ACS': len(acs_cols),
        'Engineered': len(engineered_cols),
    }
    source_counts = {k: v for k, v in source_counts.items() if v > 0}
    colors = ['#3498db', '#e74c3c', '#2ecc71', '#f39c12']
    axes[0].bar(source_counts.keys(), source_counts.values(),
                color=colors[:len(source_counts)], edgecolor='white')
    axes[0].set_title('Features by Source')
    axes[0].set_ylabel('Number of Features')
    for i, (k, v) in enumerate(source_counts.items()):
        axes[0].text(i, v + 0.1, str(v), ha='center', fontsize=11, fontweight='bold')

    # 2. Composite vulnerability distribution (if available)
    if 'composite_vulnerability_score' in merged.columns:
        axes[1].hist(merged['composite_vulnerability_score'], bins=60,
                     color='#3498db', edgecolor='white', alpha=0.85)
        axes[1].axvline(merged['composite_vulnerability_score'].median(),
                        color='#e74c3c', linestyle='--', linewidth=2, label='Median')
        axes[1].set_title('Composite Vulnerability Score Distribution')
        axes[1].set_xlabel('Score (0 = least vulnerable, 1 = most vulnerable)')
        axes[1].set_ylabel('ZCTA Count')
        axes[1].legend()
    else:
        axes[1].text(0.5, 0.5, 'Composite score\nnot available',
                     ha='center', va='center', transform=axes[1].transAxes, fontsize=12)
        axes[1].set_title('Composite Vulnerability Score')

    # 3. CDC PLACES feature correlations (top 8 features)
    if len(places_cols) >= 2:
        top_places = places_cols[:8]
        corr = merged[top_places].corr()
        short_names = [c.replace('places_', '') for c in top_places]
        corr.index = short_names
        corr.columns = short_names
        sns.heatmap(corr, ax=axes[2], cmap='coolwarm', center=0,
                    annot=True, fmt='.2f', annot_kws={'size': 8})
        axes[2].set_title('CDC PLACES Feature Correlations')
    else:
        axes[2].text(0.5, 0.5, 'Insufficient features\nfor correlation plot',
                     ha='center', va='center', transform=axes[2].transAxes, fontsize=12)

    plt.tight_layout()
    plt.savefig(OUTPUTS / '02b_feature_matrix_overview.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("✅ Chart saved to outputs/02b_feature_matrix_overview.png")

## Step 12: Final Dataset Summary

A full accounting of what's in the feature matrix before it gets saved.

In [ ]:
if merged is not None:
    feature_cols    = [c for c in merged.columns if c != 'ZCTA']
    places_cols     = [c for c in merged.columns if c.startswith('places_') and not c.endswith('_missing')]
    hhi_cols        = [c for c in merged.columns if c.startswith('hhi_') and not c.endswith('_missing')]
    acs_cols        = [c for c in merged.columns if c.startswith('acs_') and not c.endswith('_missing')]
    missing_flags   = [c for c in merged.columns if c.endswith('_missing')]
    engineered_cols = [c for c in merged.columns if c in ['composite_vulnerability_score', 'multi_condition_burden']]

    print("=" * 60)
    print("FINAL FEATURE MATRIX — SUMMARY")
    print("=" * 60)
    print(f"Total ZCTAs:              {len(merged):,}")
    print(f"Total columns:            {merged.shape[1]}")
    print()
    print(f"CDC PLACES features:      {len(places_cols)}")
    print(f"HHI features:             {len(hhi_cols)}")
    print(f"ACS features:             {len(acs_cols)}")
    print(f"Missingness flag columns: {len(missing_flags)}")
    print(f"Engineered features:      {len(engineered_cols)}")
    print()
    print(f"Remaining nulls:          {merged.isnull().sum().sum()}")
    print()
    print("All columns:")
    for col in sorted(merged.columns):
        dtype = str(merged[col].dtype)
        null_n = merged[col].isnull().sum()
        null_str = f"  ← {null_n} nulls" if null_n > 0 else ""
        print(f"  {col:<45} {dtype}{null_str}")

## Step 13: Save Feature Matrix

Saves the merged, cleaned, engineered dataset to `data/processed/`. This file is the input for the label construction notebook and, ultimately, the XGBoost model.

In [ ]:
if merged is not None:
    output_path = DATA_PROCESSED / 'sustainable_features_merged.csv'
    merged.to_csv(output_path, index=False)

    file_size = output_path.stat().st_size / 1024 / 1024
    print(f"✅ Saved: {output_path}")
    print(f"   Size:    {file_size:.2f} MB")
    print(f"   Rows:    {len(merged):,}")
    print(f"   Columns: {merged.shape[1]}")
    print()
    print("This file is the feature matrix input for:")
    print("   → 03_label_construction.ipynb  (build the Y variable)")
    print("   → 04_model_training.ipynb       (train the XGBoost model)")
else:
    print("⚠️  Nothing to save — merged dataframe is None.")
    print("   Complete Steps 2–7 first.")

---

## ✅ What You Just Built

You merged multiple federal datasets into a single, analysis-ready feature matrix at the ZCTA level. Every row is a ZIP code. Every column is a feature the XGBoost model can learn from.

---

## 🔜 Next Notebooks

| Notebook | Purpose |
|---|---|
| `03_label_construction.ipynb` | Build the Y variable — which ZCTAs experienced elevated heat illness? |
| `04_model_training.ipynb` | Train the XGBoost classifier on the feature matrix + labels |
| `05_model_evaluation.ipynb` | AUC-ROC, recall, precision-recall curve; compare against HHI baseline |
| `06_shap_explainability.ipynb` | SHAP values — understand *why* each ZIP code was flagged |

---

*SustAInable — Neighborhood Heat Illness Risk Prediction*  
*Equitech Futures Civic Tech Institute, CTI 2026*